# SAS to Databricks: Credit Pricing Linear and Polynomial Regression

This beginner notebook shows how a simple SAS credit-pricing regression workflow maps to Databricks and PySpark.

You will learn how to:
- Create simple credit pricing data
- Visualize the relationship between credit score and interest rate
- Train a linear regression model
- Add polynomial features for a curved pricing pattern
- Compare model performance on test data


## 1. Import Libraries

Databricks provides the `spark` session. We import PySpark ML classes for modeling and Matplotlib for charts.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, VectorAssembler
from pyspark.ml.regression import LinearRegression

warnings.filterwarnings("ignore")

print("Libraries are ready")
print(f"Spark version: {spark.version}")


## 2. Create Simple Credit Pricing Data

The target column is `interest_rate_percent`. Lower credit scores usually receive higher interest rates. The relationship is curved so polynomial regression has something useful to learn.


In [ ]:
np.random.seed(42)
row_count = 200

credit_score = np.random.randint(540, 851, row_count)
score_risk = np.maximum((720 - credit_score) / 100, 0)

interest_rate_percent = (
    4.0
    + 2.0 * score_risk
    + 2.8 * score_risk**2
    + np.random.normal(0, 0.45, row_count)
).clip(3.0, 24.0)

credit_data = pd.DataFrame({
    "credit_score": credit_score,
    "interest_rate_percent": interest_rate_percent,
})

credit_pricing = spark.createDataFrame(credit_data)

print(f"Rows: {credit_pricing.count()}")
credit_pricing.show(5)
credit_pricing.describe().show()


## 3. SAS Reference: Create Credit Pricing Data

In SAS, a `DATA` step can create the same kind of training table.

```sas
DATA credit_pricing;
  DO customer_id = 1 TO 200;
    credit_score = 540 + FLOOR(RAND('UNIFORM') * 311);
    score_risk = MAX((720 - credit_score) / 100, 0);
    interest_rate_percent = 4.0 + 2.0*score_risk + 2.8*score_risk*score_risk
                            + RAND('NORMAL') * 0.45;
    OUTPUT;
  END;
RUN;

PROC PRINT DATA=credit_pricing (OBS=5);
RUN;
```


## 4. Visualize the Credit Pricing Data

Each dot is one borrower. The chart helps us see that interest rates rise faster when credit scores are lower.


In [ ]:
plot_data = credit_pricing.select("credit_score", "interest_rate_percent").toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(plot_data["credit_score"], plot_data["interest_rate_percent"], alpha=0.7)
plt.title("Interest Rate by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Interest Rate (%)")
plt.grid(True)
plt.show()


## 5. Prepare Features and Split Data

PySpark ML expects model inputs in one vector column named `features`. We also split rows into train and test sets so evaluation is realistic.


In [ ]:
feature_columns = ["credit_score"]
target_column = "interest_rate_percent"

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
model_data = assembler.transform(credit_pricing).withColumnRenamed(target_column, "label")

train_data, test_data = model_data.randomSplit([0.7, 0.3], seed=42)

print(f"Train rows: {train_data.count()}")
print(f"Test rows: {test_data.count()}")
model_data.select("credit_score", "features", "label").show(5, truncate=False)


## 6. Train a Linear Regression Model

This is the PySpark version of a simple SAS `PROC REG` model.


In [ ]:
linear_regression = LinearRegression(featuresCol="features", labelCol="label")
linear_model = linear_regression.fit(train_data)

linear_train_predictions = linear_model.transform(train_data)
linear_test_predictions = linear_model.transform(test_data)

print("Linear regression model")
print(f"Intercept: {linear_model.intercept:,.4f}")
print(f"credit_score coefficient: {linear_model.coefficients[0]:,.6f}")

linear_test_predictions.select("credit_score", "label", "prediction").show(5)


## 7. SAS Reference: Linear Credit Pricing Regression

SAS can train the same one-feature model with `PROC REG`.

```sas
PROC REG DATA=credit_pricing;
  MODEL interest_rate_percent = credit_score;
  OUTPUT OUT=linear_predictions P=predicted_rate;
RUN;
QUIT;
```


## 8. Train a Polynomial Regression Model

Polynomial regression adds a squared credit-score effect. This helps the model learn a curved pricing relationship.


In [ ]:
polynomial_expansion = PolynomialExpansion(
    degree=2,
    inputCol="features",
    outputCol="polynomial_features",
)

polynomial_train_data = polynomial_expansion.transform(train_data)
polynomial_test_data = polynomial_expansion.transform(test_data)

polynomial_regression = LinearRegression(
    featuresCol="polynomial_features",
    labelCol="label",
)
polynomial_model = polynomial_regression.fit(polynomial_train_data)
polynomial_train_predictions = polynomial_model.transform(polynomial_train_data)
polynomial_test_predictions = polynomial_model.transform(polynomial_test_data)

print("Polynomial regression model")
print(f"Intercept: {polynomial_model.intercept:,.4f}")
print(f"credit_score coefficient: {polynomial_model.coefficients[0]:,.6f}")
print(f"credit_score squared coefficient: {polynomial_model.coefficients[1]:,.8f}")


## 9. SAS Reference: Polynomial Credit Pricing Regression

SAS can create the squared feature manually before regression.

```sas
DATA credit_pricing_poly;
  SET credit_pricing;
  credit_score_sq = credit_score * credit_score;
RUN;

PROC REG DATA=credit_pricing_poly;
  MODEL interest_rate_percent = credit_score credit_score_sq;
  OUTPUT OUT=poly_predictions P=predicted_rate;
RUN;
QUIT;
```


## 10. Visualize Linear and Polynomial Pricing Curves

The linear model creates a straight line. The polynomial model can bend, which is often closer to real risk-based pricing.


In [ ]:
credit_score_grid = pd.DataFrame({"credit_score": np.arange(540, 851, 5)})
curve_data = spark.createDataFrame(credit_score_grid)
curve_features = assembler.transform(curve_data)
curve_polynomial_features = polynomial_expansion.transform(curve_features)

linear_curve = linear_model.transform(curve_features).select("credit_score", "prediction").toPandas()
polynomial_curve = polynomial_model.transform(curve_polynomial_features).select("credit_score", "prediction").toPandas()

plt.figure(figsize=(9, 5))
plt.scatter(plot_data["credit_score"], plot_data["interest_rate_percent"], alpha=0.25, label="Actual")
plt.plot(linear_curve["credit_score"], linear_curve["prediction"], label="Linear prediction")
plt.plot(polynomial_curve["credit_score"], polynomial_curve["prediction"], label="Polynomial prediction")
plt.title("Credit Pricing Curve")
plt.xlabel("Credit Score")
plt.ylabel("Interest Rate (%)")
plt.legend()
plt.grid(True)
plt.show()


## 11. Compare the Models

RMSE and MAE are error values, so lower is better. R2 is a fit score, so higher is better.


In [ ]:
rmse_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

linear_rmse = rmse_evaluator.evaluate(linear_test_predictions)
linear_mae = mae_evaluator.evaluate(linear_test_predictions)
linear_r2 = r2_evaluator.evaluate(linear_test_predictions)

polynomial_rmse = rmse_evaluator.evaluate(polynomial_test_predictions)
polynomial_mae = mae_evaluator.evaluate(polynomial_test_predictions)
polynomial_r2 = r2_evaluator.evaluate(polynomial_test_predictions)

print(f"{'Metric':<10} {'Linear':>15} {'Polynomial':>15}")
print("-" * 42)
print(f"{'RMSE':<10} {linear_rmse:>15.4f} {polynomial_rmse:>15.4f}")
print(f"{'MAE':<10} {linear_mae:>15.4f} {polynomial_mae:>15.4f}")
print(f"{'R2':<10} {linear_r2:>15.4f} {polynomial_r2:>15.4f}")


## 12. Visualize Model Error

Smaller RMSE and MAE bars mean the model has lower prediction error.


In [ ]:
metric_names = ["RMSE", "MAE"]
linear_scores = [linear_rmse, linear_mae]
polynomial_scores = [polynomial_rmse, polynomial_mae]
x_positions = np.arange(len(metric_names))
bar_width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x_positions - bar_width / 2, linear_scores, bar_width, label="Linear")
plt.bar(x_positions + bar_width / 2, polynomial_scores, bar_width, label="Polynomial")
plt.title("Linear vs Polynomial Credit Pricing Error")
plt.xticks(x_positions, metric_names)
plt.ylabel("Error")
plt.legend()
plt.grid(axis="y")
plt.show()


## 13. SAS to Databricks Mapping

| Task | SAS | Databricks |
| --- | --- | --- |
| Create credit data | `DATA ... RUN;` | Spark DataFrame |
| Explore data | `PROC PRINT`, `PROC MEANS` | `show()`, `describe()` |
| Linear pricing model | `PROC REG` | `LinearRegression` |
| Curved pricing term | Manual squared column | `PolynomialExpansion` |
| Score applications | `OUTPUT OUT=` | `model.transform()` |
| Visual review | `PROC SGPLOT` | Matplotlib |

Start with the linear model first. Use polynomial regression when a chart or test metrics show that pricing behavior is curved.
